# Hebrew TrOCR — multi-model comparison on `dataset_matan`

Runs **CER / WER / BLEU** for three models on the same out-of-domain test set
(`cyttic/trocr-hebrew-matan`, see `to_parquet_pairs.py` / `push_to_hf_matan.py`
in the repo) and reports them side by side:

| Model | Stage |
|-------|-------|
| `cyttic/trocr-hebrew-synthetic-cont` | Experiment 2 — synthetic pretrain (encoder frozen) |
| `cyttic/trocr-hebrew-synthetic-cont-unfrozen` | Experiment 3 — synthetic continuation, encoder unfrozen |
| `cyttic/trocr-hebrew-finetuned` | Human-finetuned model |

**Notebook settings: GPU T4 x2 + Internet ON.**

How the parallelism works: for *each* model, the matan test set is split into
`N_GPU` contiguous chunks. Each chunk gets its own model copy on its own CUDA
device and runs in its own thread (the same pattern as
`experiment_3_synthetic_unfrozen/kaggle_eval_full_2gpu.py`).
`torch.nn.DataParallel` is **not** used because `.generate()` (beam search)
doesn't split cleanly across replicas — independent copies + independent data
slices is the simple, correct pattern for parallel inference. CUDA kernels
release the GIL while the GPU works, so the two threads' `generate()` calls
genuinely overlap on the two T4s.

Models are evaluated **one at a time** (each gets the full 2-GPU split in turn)
so that only one model's weights occupy VRAM at once — three full models
wouldn't fit on 2x16GB T4s simultaneously.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jiwer", "sacrebleu"], check=False)

In [ ]:
import gc
import math
import time
import threading
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
import pandas as pd
import jiwer
import sacrebleu
from PIL import Image, ImageOps
from datasets import load_dataset
from transformers import VisionEncoderDecoderModel, AutoTokenizer
from IPython.display import display
import matplotlib.pyplot as plt

In [ ]:
MODELS = [
    "cyttic/trocr-hebrew-synthetic-cont",
    "cyttic/trocr-hebrew-synthetic-cont-unfrozen",
    "cyttic/trocr-hebrew-finetuned",
]
DATASET = "cyttic/trocr-hebrew-matan"   # see push_to_hf_matan.py
SPLIT   = "test"    # converted with --test-frac -> writer-level held-out split
BEAMS   = 4
BATCH   = 16        # PER-GPU batch; T4 16GB + fp16 (drop to 8 on OOM)
MAXLEN  = 128
LIMIT   = 0         # 0 = full split; set e.g. 2000 for a quick check

N_GPU = torch.cuda.device_count()
assert N_GPU >= 1, "no CUDA GPU found -- enable GPU T4 x2 in notebook settings"
print(f"GPUs visible: {N_GPU}")

In [ ]:
# -- HebrewBlockProcessor (inlined; must match the repo's block_processor.py) --
class HebrewBlockProcessor:
    TARGET_HEIGHT = 64
    CONTAINER_SIZE = 384
    IMAGE_MEAN = [0.5, 0.5, 0.5]
    IMAGE_STD = [0.5, 0.5, 0.5]

    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        return {"pixel_values": torch.stack([self._process(i) for i in images])}

    def _process(self, image):
        image = image.convert("RGB")
        image = ImageOps.mirror(image)
        w, h = image.size
        new_w = max(1, round(w * self.TARGET_HEIGHT / h))
        image = image.resize((new_w, self.TARGET_HEIGHT), Image.LANCZOS)
        container = Image.new("RGB", (self.CONTAINER_SIZE, self.CONTAINER_SIZE), (255, 255, 255))
        arr = np.array(image)
        src_x, dest_x, dest_y = 0, 0, 0
        while src_x < new_w and dest_y < self.CONTAINER_SIZE:
            chunk_w = min(new_w - src_x, self.CONTAINER_SIZE - dest_x)
            chunk = Image.fromarray(arr[:, src_x:src_x + chunk_w])
            container.paste(chunk, (dest_x, dest_y))
            src_x += chunk_w
            dest_x += chunk_w
            if dest_x >= self.CONTAINER_SIZE:
                dest_x = 0
                dest_y += self.TARGET_HEIGHT
        t = torch.tensor(np.array(container), dtype=torch.float32).permute(2, 0, 1) / 255.0
        mean = torch.tensor(self.IMAGE_MEAN).view(3, 1, 1)
        std = torch.tensor(self.IMAGE_STD).view(3, 1, 1)
        return (t - mean) / std

In [ ]:
# -- load the matan test set ONCE; every model is evaluated on the exact same rows --
ds = load_dataset(DATASET, split=SPLIT)
if LIMIT:
    ds = ds.select(range(LIMIT))
N = len(ds)
print(f"{DATASET}[{SPLIT}]  samples: {N}  | beams={BEAMS} | batch/GPU={BATCH} | GPUs={N_GPU}", flush=True)

chunk_size = math.ceil(N / N_GPU)
chunks = [(i * chunk_size, min((i + 1) * chunk_size, N)) for i in range(N_GPU)]
print("chunks (lo, hi):", chunks, flush=True)

print_lock = threading.Lock()

In [ ]:
# -- per-(model, GPU) worker: own model copy, own slice of the data --
def run_on_gpu(model_id, gpu_id, lo, hi):
    device = f"cuda:{gpu_id}"
    tag = f"[{model_id.split('/')[-1]} | gpu{gpu_id}]"

    model = VisionEncoderDecoderModel.from_pretrained(model_id).to(device).eval().half()
    tok = AutoTokenizer.from_pretrained(model_id)
    proc = HebrewBlockProcessor()
    model.generation_config.decoder_start_token_id = tok.cls_token_id
    model.generation_config.pad_token_id = tok.pad_token_id
    model.generation_config.eos_token_id = tok.sep_token_id
    model.generation_config.max_new_tokens = None

    sub = ds.select(range(lo, hi))
    n = len(sub)
    refs, hyps = [], []
    t0 = time.time()
    for start in range(0, n, BATCH):
        batch = sub[start:start + BATCH]
        imgs = [im.convert("RGB") for im in batch["image"]]
        pv = proc(imgs)["pixel_values"].to(device, dtype=model.dtype)
        with torch.no_grad():
            ids = model.generate(pv, num_beams=BEAMS, max_length=MAXLEN)
        hyps.extend(tok.batch_decode(ids, skip_special_tokens=True))
        refs.extend(batch["text"])
        done = min(start + BATCH, n)
        if done % (BATCH * 20) == 0 or done == n:
            el = time.time() - t0
            rate = done / el
            eta = (n - done) / rate if rate else 0
            with print_lock:
                print(f"  {tag} {done}/{n}  | {rate:.1f} img/s | elapsed {el/60:.1f}m | ETA {eta/60:.1f}m", flush=True)

    with print_lock:
        print(f"  {tag} FINISHED  ({n} samples in {(time.time()-t0)/60:.1f}m)", flush=True)

    # free this GPU's copy before the next model loads -- 3 full models won't
    # fit in VRAM at once on 2x16GB T4s, so we evaluate them strictly in turn
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return refs, hyps

In [ ]:
# -- evaluate one model: split across both GPUs in parallel, then score --
def evaluate_model(model_id):
    print(f"\n{'='*70}\nEvaluating {model_id}\n  on {DATASET}[{SPLIT}]  (N={N}, beams={BEAMS}, {N_GPU}x T4 parallel)\n{'='*70}", flush=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_GPU) as ex:
        futures = [ex.submit(run_on_gpu, model_id, gpu_id, lo, hi) for gpu_id, (lo, hi) in enumerate(chunks)]
        results = [f.result() for f in futures]

    # results are in chunk order -> concatenation preserves correct ref/hyp pairing
    refs, hyps = [], []
    for r, h in results:
        refs.extend(r)
        hyps.extend(h)
    elapsed = time.time() - t0

    cer = jiwer.cer(refs, hyps)
    wer = jiwer.wer(refs, hyps)
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    exact = sum(r.strip() == h.strip() for r, h in zip(refs, hyps)) / len(refs)

    print(f"\n{model_id}  |  N={len(refs)}  |  CER {cer*100:.2f}%  WER {wer*100:.2f}%  "
          f"BLEU {bleu:.2f}  |  exact {exact*100:.2f}%  |  {elapsed/60:.1f} min wall-clock", flush=True)

    return {"model": model_id, "N": len(refs), "CER": cer, "WER": wer, "BLEU": bleu,
            "exact": exact, "minutes": elapsed / 60, "refs": refs, "hyps": hyps}

In [ ]:
# -- run all three models, one at a time (each uses both GPUs) --
all_results = [evaluate_model(model_id) for model_id in MODELS]
print("\nAll models evaluated.")

In [ ]:
# -- friendly display names, matching the repo's experiment-folder naming --
DISPLAY_NAME = {
    "cyttic/trocr-hebrew-finetuned":               "exp1_human",
    "cyttic/trocr-hebrew-synthetic-cont":          "exp2_synth",
    "cyttic/trocr-hebrew-synthetic-cont-unfrozen": "exp3_unfrozen",
}

def nice(model_id):
    return DISPLAY_NAME.get(model_id, model_id)

for r in all_results:
    r["label"] = nice(r["model"])

print({r["model"]: r["label"] for r in all_results})

In [ ]:
# -- comparison table --
summary = pd.DataFrame([
    {"Model": r["label"],
     "Samples": f"{r['N']:,}",
     "CER": f"{r['CER']*100:.2f}%",
     "WER": f"{r['WER']*100:.2f}%",
     "BLEU": f"{r['BLEU']:.2f}",
     "Exact-match": f"{r['exact']*100:.2f}%",
     "Minutes": f"{r['minutes']:.1f}"}
    for r in all_results
])

display(
    summary.style.hide(axis="index")
    .set_caption(f"Hebrew TrOCR — model comparison on {DATASET}[{SPLIT}]  (N={all_results[0]['N']:,}, beam={BEAMS}, {N_GPU}x T4 parallel)")
    .set_properties(**{"font-size": "15px", "text-align": "left", "padding": "6px 22px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "17px"), ("font-weight", "bold"),
                                          ("padding", "10px"), ("color", "#222")]},
        {"selector": "th", "props": [("background-color", "#1f77b4"), ("color", "white"),
                                     ("font-size", "15px"), ("text-align", "left"),
                                     ("padding", "6px 22px")]},
        {"selector": "td", "props": [("border-bottom", "1px solid #ddd")]},
    ])
)

In [ ]:
# -- grouped bar chart: CER & WER per model --
labels = [r["label"] for r in all_results]
cer_vals = [r["CER"] * 100 for r in all_results]
wer_vals = [r["WER"] * 100 for r in all_results]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - width/2, cer_vals, width, label="CER", color="#1f77b4")
b2 = ax.bar(x + width/2, wer_vals, width, label="WER", color="#d62728")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{b.get_height():.1f}%",
                ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("error rate (%)")
ax.set_title(f"{DATASET}[{SPLIT}] — model comparison  (N={all_results[0]['N']:,}, beam={BEAMS}, {N_GPU}x T4 parallel)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# -- sample predictions per model (3 best + 3 worst by per-line CER) --
for r in all_results:
    refs, hyps = r["refs"], r["hyps"]
    per_cer = [jiwer.cer(rf, hp) for rf, hp in zip(refs, hyps)]
    order = sorted(range(len(refs)), key=lambda i: per_cer[i])
    pick = order[:3] + order[-3:]
    tbl = pd.DataFrame(
        [[refs[i], hyps[i], f"{per_cer[i]*100:.0f}%", "\u2713" if refs[i].strip() == hyps[i].strip() else ""]
         for i in pick],
        columns=["Ground truth", "Prediction", "CER", "Match"])
    display(
        tbl.style.hide(axis="index")
        .set_caption(f"{r['label']} ({r['model']}) — sample predictions (3 best + 3 worst)")
        .set_properties(subset=["Ground truth", "Prediction"], **{"text-align": "right", "font-size": "14px"})
        .set_properties(subset=["CER", "Match"], **{"text-align": "center"})
        .set_table_styles([
            {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold"), ("padding", "8px")]},
            {"selector": "th", "props": [("background-color", "#444"), ("color", "white"), ("padding", "5px 14px")]},
        ])
    )

In [ ]:
# -- run this LAST, after the evaluation finished (reuses all_results) --
DISPLAY_NAME = {
    "cyttic/trocr-hebrew-finetuned":               "exp1_human",
    "cyttic/trocr-hebrew-synthetic-cont":          "exp2_synth",
    "cyttic/trocr-hebrew-synthetic-cont-unfrozen": "exp3_unfrozen",
}
for r in all_results:
    r["label"] = DISPLAY_NAME.get(r["model"], r["model"])

summary = pd.DataFrame([
    {"Model": r["label"],
     "Samples": f"{r['N']:,}",
     "CER": f"{r['CER']*100:.2f}%",
     "WER": f"{r['WER']*100:.2f}%",
     "BLEU": f"{r['BLEU']:.2f}",
     "Exact-match": f"{r['exact']*100:.2f}%"}
    for r in all_results
])

styled = summary.style.hide(axis="index")
styled = styled.set_caption("Model comparison on dataset_matan")
display(styled)

In [ ]:
# -- bar chart: CER & WER per model, using the friendly labels --
labels = [r["label"] for r in all_results]
cer_vals = [r["CER"] * 100 for r in all_results]
wer_vals = [r["WER"] * 100 for r in all_results]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - width/2, cer_vals, width, label="CER", color="#1f77b4")
b2 = ax.bar(x + width/2, wer_vals, width, label="WER", color="#d62728")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{b.get_height():.1f}%",
                ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("error rate (%)")
ax.set_title("dataset_matan -- model comparison (CER / WER)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# -- bar chart: BLEU per model, using the friendly labels --
bleu_vals = [r["BLEU"] for r in all_results]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, bleu_vals, color="#2ca02c", width=0.5)
for b, v in zip(bars, bleu_vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v:.1f}", ha="center", fontweight="bold")
ax.set_ylabel("BLEU")
ax.set_title("dataset_matan -- model comparison (BLEU)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# -- sample predictions per model (3 best + 3 worst by per-line CER) --
for r in all_results:
    refs, hyps = r["refs"], r["hyps"]
    per_cer = [jiwer.cer(rf, hp) for rf, hp in zip(refs, hyps)]
    order = sorted(range(len(refs)), key=lambda i: per_cer[i])
    pick = order[:3] + order[-3:]
    rows = []
    for i in pick:
        match = "yes" if refs[i].strip() == hyps[i].strip() else ""
        rows.append([refs[i], hyps[i], f"{per_cer[i]*100:.0f}%", match])
    tbl = pd.DataFrame(rows, columns=["Ground truth", "Prediction", "CER", "Match"])
    caption = r["label"] + " (" + r["model"] + ") -- sample predictions"
    styled_tbl = tbl.style.hide(axis="index")
    styled_tbl = styled_tbl.set_caption(caption)
    display(styled_tbl)